# K-Bound Master Guide — Theory → Proof → Code → Results

**Start here.** This notebook is the single entry point for understanding how the K-Bound
paper connects to this repository.

| Layer | Where |
|-------|-------|
| Papers | `docs/research/kbound/kbound_short.pdf`, `kbound.pdf` |
| Status | `docs/research/kbound/PROJECT_STATUS_AND_OPEN_PROBLEMS.md` |
| Deep map | `docs/research/kbound/THEORY_TO_CODE_MAP.md` |
| Claims | `docs/research/kbound/claim_ledger.json` |
| Reproduce | `bash docs/research/kbound/scripts/reproduce_submission.sh` |

**What you will see below:**
1. Repo layout and notebook curriculum
2. Theory spine with validators
3. Headline empirical numbers loaded from locked JSON
4. How the certificate code implements the theorem
5. Commands to rerun anything

> Notebooks `01`–`09` are topic deep-dives; some predate POEM/AETTA WIN (June 2026).
> This master guide is kept current with the freeze gate.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

# Find repo root (works if cwd is repo root or notebooks/)
REPO = Path.cwd().resolve()
for _ in range(6):
    if (REPO / "docs/research/kbound/claim_ledger.json").exists():
        break
    REPO = REPO.parent
else:
    raise RuntimeError("Could not find AutoML_Flagship_V8 root")

KBOUND = REPO / "docs/research/kbound"
RES = REPO / "experiments/kbound/results"
TV = REPO / "experiments/kbound/theory_validation"

def jload(rel):
    p = REPO / rel
    return json.loads(p.read_text()) if p.exists() else None

print("repo:", REPO)
print("short paper:", (KBOUND / "kbound_short.pdf").exists())
print("claim ledger:", (KBOUND / "claim_ledger.json").exists())

## 1 · Automated complete picture (`kbound_tour.py`)

Runs the same map as `THEORY_TO_CODE_MAP.md` and checks that headline artifacts exist.

In [ ]:
tour = REPO / "docs/research/kbound/scripts/kbound_tour.py"
proc = subprocess.run([sys.executable, str(tour)], cwd=str(REPO), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode:
    print("WARN: some artifacts missing", file=sys.stderr)

## 2 · Theory spine — what is proven and how it is checked

Each row: **LaTeX label** → **proof in TeX** → **numeric validator** → **implementation**.

In [ ]:
report = json.loads(subprocess.check_output([sys.executable, str(tour), "--json"], cwd=str(REPO), text=True))

rows = []
for t in report["theory_spine"]:
    arts_ok = all(t["artifacts_ok"].values()) if t.get("artifacts_ok") else False
    rows.append({
        "theorem": t["label"],
        "artifacts_ok": arts_ok,
        "validators": len(t["validators"]),
        "claims": ", ".join(t.get("claim_ids") or []),
    })

import pandas as pd
display(pd.DataFrame(rows))

# Load one validator artifact (Le Cam impossibility)
thm1 = jload("experiments/kbound/theory_validation/results_thm1_lecam.json")
if thm1:
    w = thm1["witness"]["kbound_rule"]
    display(Markdown(
        f"**Le Cam witness (matched evidence):** abstain_frac={w['abstain_frac']:.2f}, "
        f"false_adapt={w['false_adapt_rate']:.2f} — theory forces abstention when TV→0."
    ))

## 3 · Empirical headlines — locked results that support paper claims

Numbers below are read from **committed JSON**, not recomputed in this cell.

In [ ]:
ledger = json.loads((KBOUND / "claim_ledger.json").read_text())
supported = [c for c in ledger["claims"] if c["status"] == "supported"]
display(Markdown(f"**{len(supported)} supported claims** in claim_ledger.json"))

# Stress grid
sg = jload("experiments/kbound/results/stress_grid_multiseed_v1/LOCKED_ANALYSIS_RESULTS.json")
if sg:
    tent = sg["candidates"]["tent"]
    display(Markdown(
        f"**CIFAR stress grid (Tent):** KGA regret={tent['kga_mean_regret']:.4f}, "
        f"adapt={tent['adapt_mean_regret']:.4f}, beats_both={tent.get('beats_both_robust')}"
    ))

# POEM/AETTA head-to-head
h2h = jload("experiments/kbound/results/mixed_headtohead_v1/HEADTOHEAD_RESULTS_cifar10c_tent_primary.json")
if h2h:
    r = h2h["policy_mean_regret"]
    v = h2h["headtohead"]["VERDICT"]
    display(Markdown(
        f"**POEM/AETTA head-to-head:** VERDICT={v}; "
        f"KGA={r['kga']:.4f}, POEM={r['poem']:.4f}, AETTA={r['aetta']:.4f}; "
        f"FA_u(KGA)={h2h['policy_false_adapt_rate']['kga']:.3f}"
    ))

# Mixed OOF aggregate
mix = jload("experiments/kbound/results/mixed_protocol_oof_v2/mixed_protocol_oof_v2_result.json")
if mix:
    display(Markdown(
        f"**Mixed OOF aggregate:** n={mix.get('n_conditions', mix.get('n'))}, "
        f"beats_both_robust={mix.get('beats_both_robust')}, FA_u={mix.get('false_adapt', mix.get('kga_false_adapt'))}"
    ))

## 4 · Code path — certificate implements the theorem

`kbound_pkg/kbound/certificate.py` turns evidence + OOF radius into ADAPT/FREEZE/ABSTAIN.
Scorers: `decide_kga()` in `docs/research/kbound/scripts/analysis_F.py` (LOO conformal).

In [ ]:
cert = KBOUND / "kbound_pkg/kbound/certificate.py"
analysis = KBOUND / "scripts/analysis_F.py"
for p in [cert, analysis]:
    print(f"{'OK' if p.exists() else 'MISSING'}: {p.relative_to(REPO)}")

# Gate baseline selftest (certificate vs baselines on synthetic grid)
gate = subprocess.run(
    [sys.executable, str(KBOUND / "scripts/gate_baseline_comparison.py"), "--selftest"],
    cwd=str(KBOUND), capture_output=True, text=True,
)
print(gate.stdout[-800:] if len(gate.stdout) > 800 else gate.stdout)
if gate.returncode:
    print(gate.stderr)

## 5 · Notebook curriculum (what each of the 10+ notebooks covers)

Use topic notebooks after this guide for depth on one area.

In [ ]:
display(pd.DataFrame(report["notebook_curriculum"], columns=["notebook", "description"]))

## 6 · Reproduce everything (copy-paste)

**Fast (CPU, ~2 min):** integrity tests + table macros + artifact checks.

```bash
cd /Volumes/T9/uav/AutoML_Flagship_V8
bash docs/research/kbound/scripts/reproduce_submission.sh
bash docs/research/kbound/scripts/kbound_tour.sh
```

**Headline empirics (cached, seconds):**

```bash
PY=.venv/bin/python bash experiments/kbound/poem_aetta/run_all_headtohead.sh
.venv/bin/python docs/research/kbound/scripts/mixed_stream_kbound.py
```

**Optional full 9-dataset GPU verification (many hours):**

```bash
export KB_DEVICE=mps
bash docs/research/kbound/scripts/kbtrain.sh final-all
```

**What is genuinely open (not blocking submission):** see `PROJECT_STATUS_AND_OPEN_PROBLEMS.md` §1 — gen-capacity without R1/R2, tight rates, physical camera R2.

> macOS has no `python` command — use `python3`, `.venv/bin/python`, or the `kbound_tour.sh` wrapper above.